# 📓 Notebook 03 — Preprocessing
## G19 Facial Emotion Recognition | CSE276

---

**Purpose of this notebook:**
- Demonstrate the full preprocessing pipeline on sample images.
- Visualise the effect of CLAHE contrast enhancement.
- Demonstrate data augmentation (flips, rotations, zoom, brightness).
- Confirm that the data generators are working correctly before training.

> **Prerequisite:** Run notebook 01 first.

In [ ]:
import sys, os
sys.path.insert(0, '/content/G19_Facial_Emotion_Recognition')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.data_loader import load_fer2013, EMOTION_LABELS
from src.preprocessing import preprocess_image, apply_clahe_batch, build_augmentation_layer
from src.utils import set_random_seeds

set_random_seeds(42)
DATASET_CSV = '/content/G19_Facial_Emotion_Recognition/dataset/fer2013.csv'
RESULTS_DIR = '/content/G19_Facial_Emotion_Recognition/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete.')

## 1. Load Dataset

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = load_fer2013(DATASET_CSV)
print('Dataset loaded.')
print(f'Train shape: {X_train.shape}')

## 2. Visualise CLAHE Effect

In [ ]:
import cv2

# Pick one sample image per class and show before / after CLAHE
fig, axes = plt.subplots(NUM_CLASSES := 7, 2, figsize=(6, NUM_CLASSES * 2))

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

for class_idx in range(NUM_CLASSES):
    class_indices = np.where(y_train == class_idx)[0]
    sample_img = (X_train[class_indices[0], :, :, 0] * 255).astype(np.uint8)
    enhanced   = clahe.apply(sample_img)

    axes[class_idx][0].imshow(sample_img,  cmap='gray', vmin=0, vmax=255)
    axes[class_idx][0].set_title(f'{EMOTION_LABELS[class_idx]} — Original', fontsize=8)
    axes[class_idx][0].axis('off')

    axes[class_idx][1].imshow(enhanced, cmap='gray', vmin=0, vmax=255)
    axes[class_idx][1].set_title(f'{EMOTION_LABELS[class_idx]} — CLAHE', fontsize=8)
    axes[class_idx][1].axis('off')

plt.suptitle('Original vs CLAHE-Enhanced Images', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/clahe_comparison.png', dpi=120)
plt.show()

## 3. Visualise Data Augmentation

In [ ]:
# Select a single image and show 8 random augmentations
augmentation_layer = build_augmentation_layer()

sample_image = X_train[0:1]   # shape (1, 48, 48, 1)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
axes = axes.flatten()

# Original
axes[0].imshow(sample_image[0, :, :, 0], cmap='gray')
axes[0].set_title('Original', fontsize=9)
axes[0].axis('off')

# 7 augmented versions
for i in range(1, 8):
    aug_img = augmentation_layer(sample_image, training=True)
    axes[i].imshow(aug_img[0, :, :, 0].numpy(), cmap='gray')
    axes[i].set_title(f'Augmented {i}', fontsize=9)
    axes[i].axis('off')

plt.suptitle(f'Data Augmentation Examples — {EMOTION_LABELS[y_train[0]]}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/augmentation_examples.png', dpi=120)
plt.show()

## 4. Build Keras Data Generators

In [ ]:
BATCH_SIZE = 64

# Training dataset — shuffle and batch
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = (
    train_dataset
    .shuffle(buffer_size=len(X_train), seed=42)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Validation dataset — no shuffle
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Test dataset — no shuffle
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Verify one batch
for batch_images, batch_labels in train_dataset.take(1):
    print(f'Batch images shape : {batch_images.shape}')
    print(f'Batch labels shape : {batch_labels.shape}')
    print(f'Unique labels in batch: {sorted(set(batch_labels.numpy()))}')

print('\n✅ Data generators ready for training.')

## 5. Summary

| Step | Status |
|---|---|
| CLAHE contrast enhancement | ✅ Visualised |
| Data augmentation | ✅ Visualised |
| Keras datasets created | ✅ Ready |

**Next step → Open `04_cnn_training.ipynb`**